# A daily dependency watch with LangGraph and Web Search on Amazon Bedrock AgentCore

A security team that ships software has to know, every morning, whether anything published
since yesterday affects the open source packages and base images it depends on: a new CVE,
an advisory, a malicious package in the same ecosystem, or a vulnerability added to a
known-exploited list. Reading that by hand doesn't scale, and an agent that answers from
stale or uncited sources is worse than no agent.

This notebook builds that watch as a LangGraph graph on top of the Web Search API in
[Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/). You call the API with
the AWS credentials you already have, and there's nothing to create first.

![Architecture of the dependency watch](images/agentcore_web_search_supply_chain_watch_api.svg)

What each part demonstrates:

1. **Fresh results only, enforced in code.** Every search carries a published-after bound
   set by your code from the watch window. The model never sees a date argument, so it
   can't widen it. The bound applies to each article's publication date, not to the
   incident it describes, so the branch prompt also tells the model to report only issues
   first disclosed inside the window.
2. **A source blocklist the model can't lift.** The search tool carries the blocklist on
   every request, whatever the model asks for.
3. **Parallel branches.** One branch per ecosystem, run concurrently by LangGraph, each
   with its own search client.
4. **Every finding backed by its source, and checked in code.** Branches return structured
   findings with a source URL. Code keeps a finding only if that URL came back from the
   branch's own searches, the returned text names the package, and any CVE or advisory
   ID the finding gives appears in that text.
5. **Only what's new.** A LangGraph store remembers what was already reported, so a second
   run the same day reports nothing twice.

Queries are served inside AWS and aren't sent to a third-party search engine, which
matters here: a dependency watch puts your package names and versions into its queries.

Step 7 runs the same watch through
[AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html),
for teams that want AWS to apply one administrator source policy to every search.

## Prerequisites

- An AWS account, working in a Region that offers the Web Search API (see
  [Supported AWS Regions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html)).
- AWS credentials configured as environment variables (`AWS_ACCESS_KEY_ID`,
  `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN`) or through an AWS CLI profile.
- Python 3.10 or later.
- Model access in Amazon Bedrock for the Claude model used below. Any Bedrock chat model that
  supports tool use works.
- The IAM permission in Step 1 for your identity, and `bedrock:InvokeModel`. The optional
  Step 7 needs more; it lists them.

## Cost and terms

Web Search is billed per use; see [Amazon Bedrock AgentCore pricing](https://aws.amazon.com/bedrock/agentcore/pricing/).
One run of the watch makes at most 12 searches plus the model calls. The Web Search terms
require that the source citations and links that come with each result are kept and shown in
any output that reaches an end user, and the report below does that for every row.

## Install the packages and configure the watch

In [ ]:
%pip install -q -U "boto3>=1.43.127" "bedrock-agentcore>=1.26.0" "langchain-aws[tools]>=1.9.3" langgraph langchain

In [2]:
import json
import operator
import os
import re
import time
from datetime import datetime, timedelta, timezone
from typing import Annotated, Literal, Optional, TypedDict
from urllib.parse import urlparse

import boto3
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_aws.tools import create_web_search_toolkit
from langgraph.graph import END, START, StateGraph
from langgraph.store.memory import InMemoryStore
from langgraph.types import Send
from pydantic import BaseModel, Field

# Set the Region explicitly rather than reading AWS_REGION, and pick one that offers the
# Web Search API: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html
REGION = "us-east-1"

MODEL_ID = "bedrock_converse:us.anthropic.claude-sonnet-5"
# A branch can return several findings in one structured reply, so leave room for them.
model = init_chat_model(MODEL_ID, region_name=REGION, max_tokens=8192)

# What this team ships on. Replace with your own dependency inventory.
WATCHLIST = {
    "npm": ["express", "axios", "lodash"],
    "PyPI": ["requests", "urllib3", "cryptography"],
    "container base images": ["openssl", "glibc", "zlib"],
}

# Only material published inside this look-back window can reach the report. The store in
# Step 4 stops anything being reported twice, so a daily run with a week of look-back
# still reports only what's new.
WINDOW_DAYS = 7
MAX_SEARCHES_PER_BRANCH = 4

# Domains your security team doesn't accept as evidence. The default blocks AWS's own
# marketing site, so the rule is visible in the results without naming anyone else's;
# put your team's real list here.
BLOCKLIST = ["amazon.com"]


def domain_of(url):
    host = urlparse(url).hostname or ""
    return host[4:] if host.startswith("www.") else host


def matches(url, domains):
    host = domain_of(url)
    return any(host == d or host.endswith("." + d) for d in domains)


def normalize(url):
    return url.rstrip("/").split("#")[0]

## Step 1: Grant the permission

The identity that calls the Web Search API needs one IAM action on one AWS-owned resource,
the same permission a gateway's service role uses for web search. Attach this statement to
the user or role you run the notebook as. When the watch runs on
AgentCore Runtime, the Runtime folder next to this notebook attaches it to the execution role.

In [3]:
policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": "bedrock-agentcore:InvokeWebSearch",
        "Resource": f"arn:aws:bedrock-agentcore:{REGION}:aws:tool/web-search.v1",
    }],
}
print(json.dumps(policy, indent=2))

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": "bedrock-agentcore:InvokeWebSearch",
      "Resource": "arn:aws:bedrock-agentcore:us-east-1:aws:tool/web-search.v1"
    }
  ]
}


## Step 2: Create a search tool with a fixed date window

Each research branch gets its own search tool from `create_web_search_toolkit` in
`langchain-aws`. With no gateway argument, the tool calls the Web Search API. The toolkit
options set the rules that must hold whatever the model does:

- `published_after` and `exclude_domains` are carried by every search. The model can't
  widen them.
- `model_filters=False` shows the model a query argument only.
- `max_searches` caps the searches per branch, so the cost of a run is bounded.
- `return_sources=True` returns the results twice: as text for the model, and as structured
  data (an artifact) that the graph uses later to check each finding against its source.

Each branch creates its **own** toolkit, so the branches' searches run concurrently.

In [4]:
def make_search_tool(window_start, route="api"):
    # On the gateway route (Step 7), the target applies the blocklist.
    route_args = ({"exclude_domains": BLOCKLIST} if route == "api"
                  else {"gateway_id": GATEWAY_ID, "target_name": TARGET_NAME})
    toolkit, tools = create_web_search_toolkit(
        region=REGION,
        **route_args,
        published_after=window_start.strftime("%Y-%m-%dT%H:%M:%SZ"),
        model_filters=False,
        max_searches=MAX_SEARCHES_PER_BRANCH,
        return_sources=True,
    )
    search_tool = tools[0]
    search_tool.handle_tool_error = True
    return search_tool, toolkit


window_start = datetime.now(timezone.utc) - timedelta(days=WINDOW_DAYS)
probe_tool, probe_toolkit = make_search_tool(window_start)
print("arguments the model sees:", list(probe_tool.tool_call_schema.model_json_schema()["properties"]))
message = probe_tool.invoke({"type": "tool_call", "id": "probe", "name": probe_tool.name,
                             "args": {"query": "known exploited vulnerabilities added"}})
print(f"\n{len(message.artifact)} results for a window starting {window_start:%Y-%m-%d %H:%M} UTC:")
for s in message.artifact[:5]:
    print(f"   {s['published_at']} | {domain_of(s['url'])}")
probe_toolkit.close()

arguments the model sees: ['query']



24 results for a window starting 2026-11-03 23:41 UTC:
   2026-11-09T12:00:00+00:00 | cisa.gov
   2026-11-10T12:00:00+00:00 | cisa.gov
   2026-11-09T12:00:00+00:00 | cisa.gov
   2026-11-10T11:52:00+00:00 | thehackernews.com
   2026-11-10T21:32:00+00:00 | securityaffairs.com


## Step 3: Define the findings schema and the branch prompt

Each branch is a small agent that searches, then returns a list of findings in a fixed
schema rather than prose. Every finding must name the package it affects and the URL of
the source it came from.

In [5]:
class Finding(BaseModel):
    package: str = Field(description="The watched package or image this affects, exactly as named in the watchlist.")
    title: str = Field(description="One line: what happened.")
    identifier: Optional[str] = Field(default=None, description="CVE or advisory ID if the source gives one.")
    severity: Literal["critical", "high", "medium", "low", "unknown"] = Field(
        default="unknown", description="The severity the source states. Use unknown if the source doesn't state one.")
    actively_exploited: bool = Field(description="True only if the source says it is exploited in the wild.")
    action: str = Field(description="What the team should do, in one sentence of 25 words or fewer. Vendor-neutral: never recommend a commercial product or service.")
    source_url: str = Field(description="URL of the search result this finding came from. Must be one you were given.")
    published: Optional[str] = Field(default=None, description="Publication date the search result reported.")


class EcosystemFindings(BaseModel):
    findings: list[Finding] = Field(description="Empty if nothing in the window affects the watched packages.")


BRANCH_PROMPT = """You watch one software ecosystem for a security team.
Search for anything published in the watch window that affects the watched packages:
new CVEs, security advisories, malicious or compromised packages, and vulnerabilities
reported as exploited. Run at most four searches.

Report a finding only when a search result is about one of the watched packages itself:
a vulnerability, advisory or compromise in that package. A result that only mentions the
package in passing, or is about a different package that depends on it, is not a finding.
Use the exact source URL from the search results. Take severity only from what the source
states. Keep actions vendor-neutral, such as upgrading, patching or checking configuration,
and never recommend a commercial product or service. If nothing in the results affects the
watched packages, return no findings. Never fill gaps from memory.

Report only issues first disclosed inside the watch window. A recent result that describes
an older incident as background is not a finding."""

## Step 4: Build the graph

`fan_out` sends one branch per ecosystem with LangGraph's `Send`, so the branches run in
parallel. Each branch runs its agent and then **checks every finding against its source**:
the URL must have come back from that branch's searches, the returned text must name the
package as a whole word, any CVE or advisory ID must appear in that text, and the source's
publication date must fall inside the watch window. A missing or unparseable date counts as
outside. Findings that fail are dropped and counted by reason. `keep_new`
compares findings with a store of what was already reported, and `report` builds the
table. The `route` in the state says which way the branches search, so Step 7 can reuse the
same graph through AgentCore Gateway.

In [6]:
class WatchState(TypedDict):
    watchlist: dict
    window_start: str
    route: str
    findings: Annotated[list, operator.add]
    branch_stats: Annotated[list, operator.add]
    new_findings: list
    report: str


class BranchState(TypedDict):
    ecosystem: str
    packages: list
    window_start: str
    route: str


def fan_out(state: WatchState):
    return [
        Send("research", {"ecosystem": eco, "packages": pkgs, "window_start": state["window_start"],
                          "route": state["route"]})
        for eco, pkgs in state["watchlist"].items()
    ]


def research(branch: BranchState):
    started = time.perf_counter()
    window_start = datetime.fromisoformat(branch["window_start"])
    search_tool, toolkit = make_search_tool(window_start, branch["route"])
    try:
        agent = create_agent(model, tools=[search_tool], system_prompt=BRANCH_PROMPT,
                             response_format=ToolStrategy(EcosystemFindings))
        result = agent.invoke({"messages": [{"role": "user", "content":
            f"Ecosystem: {branch['ecosystem']}. Watched packages: {', '.join(branch['packages'])}. "
            f"Watch window starts {window_start:%Y-%m-%d %H:%M} UTC."}]})
    finally:
        toolkit.close()

    returned = {normalize(s["url"]): s
                for m in result["messages"] if m.type == "tool" for s in (m.artifact or [])}
    searches = sum(1 for m in result["messages"] if m.type == "tool" and m.artifact)
    kept, dropped = [], {"url_not_returned": 0, "package_not_in_source": 0, "id_not_in_source": 0,
                         "date_outside_window": 0}
    for finding in result["structured_response"].findings:
        source = returned.get(normalize(finding.source_url))
        evidence = ((source or {}).get("title") or "") + " " + ((source or {}).get("text") or "")
        published = datetime.fromisoformat(source["published_at"]) if source and source["published_at"] else None
        if source is None:
            dropped["url_not_returned"] += 1
        elif not re.search(rf"(?<![\w.-]){re.escape(finding.package)}(?![\w-])", evidence, re.IGNORECASE):
            dropped["package_not_in_source"] += 1
        elif finding.identifier and finding.identifier.lower() not in evidence.lower():
            dropped["id_not_in_source"] += 1
        elif published is None or published < window_start:
            # The search already asked for published-after; this makes it a hard rule.
            dropped["date_outside_window"] += 1
        else:
            kept.append({**finding.model_dump(), "ecosystem": branch["ecosystem"],
                         "source_published": source["published_at"]})
    return {
        "findings": kept,
        "branch_stats": [{
            "ecosystem": branch["ecosystem"], "searches": searches, "sources_returned": len(returned),
            "findings_kept": len(kept), "dropped": dropped,
            "blocked_domain_sources": sum(matches(u, BLOCKLIST) for u in returned),
            "seconds": round(time.perf_counter() - started, 1),
        }],
    }


def finding_key(f):
    return (f.get("identifier") or normalize(f["source_url"])).lower() + "|" + f["package"].lower()


def keep_new(state: WatchState, *, store):
    new = []
    for f in state["findings"]:
        key = finding_key(f)
        if store.get(("reported",), key) is None:
            store.put(("reported",), key, {"first_seen": datetime.now(timezone.utc).isoformat(), **f})
            new.append(f)
    return {"new_findings": new}


def report(state: WatchState):
    rows = sorted(state["new_findings"], key=lambda f: (not f["actively_exploited"],
                  ["critical", "high", "medium", "low", "unknown"].index(f["severity"])))
    if not rows:
        return {"report": "No new findings in the watch window."}
    lines = ["| Package | What happened | ID | Severity | Exploited | Action | Source |",
             "|---|---|---|---|---|---|---|"]
    for f in rows:
        lines.append(f"| {f['package']} ({f['ecosystem']}) | {f['title']} | {f.get('identifier') or ''} | "
                     f"{f['severity']} | {'yes' if f['actively_exploited'] else 'no'} | {f['action']} | "
                     f"[{domain_of(f['source_url'])}]({f['source_url']}) |")
    return {"report": "\n".join(lines)}


builder = StateGraph(WatchState)
builder.add_node("research", research)
builder.add_node("keep_new", keep_new)
builder.add_node("report", report)
builder.add_conditional_edges(START, fan_out, ["research"])
builder.add_edge("research", "keep_new")
builder.add_edge("keep_new", "report")
builder.add_edge("report", END)

# InMemoryStore keeps this notebook self-contained. For a scheduled job, use a durable
# store with exact key lookups, such as DynamoDBStore from langgraph-checkpoint-aws.
store = InMemoryStore()
watch = builder.compile(store=store)
print(watch.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	research(research)
	keep_new(keep_new)
	report(report)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> research;
	keep_new --> report;
	research --> keep_new;
	report --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 5: Run the watch

In [7]:
def run_watch(graph=watch, route="api"):
    started = time.perf_counter()
    window_start = datetime.now(timezone.utc) - timedelta(days=WINDOW_DAYS)
    state = graph.invoke({"watchlist": WATCHLIST, "window_start": window_start.isoformat(),
                          "route": route, "findings": [], "branch_stats": []})
    return state, round(time.perf_counter() - started, 1)


state, wall = run_watch()
print(f"wall time {wall}s across {len(state['branch_stats'])} parallel branches "
      f"(sum of branch times {sum(b['seconds'] for b in state['branch_stats']):.1f}s)\n")
for b in state["branch_stats"]:
    print(b)
print(f"\nfindings kept: {len(state['findings'])}, new this run: {len(state['new_findings'])}")

wall time 21.7s across 3 parallel branches (sum of branch times 39.0s)

{'ecosystem': 'npm', 'searches': 2, 'sources_returned': 37, 'findings_kept': 0, 'dropped': {'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 0, 'date_outside_window': 0}, 'blocked_domain_sources': 0, 'seconds': 4.4}
{'ecosystem': 'PyPI', 'searches': 4, 'sources_returned': 59, 'findings_kept': 0, 'dropped': {'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 2, 'date_outside_window': 0}, 'blocked_domain_sources': 0, 'seconds': 12.9}
{'ecosystem': 'container base images', 'searches': 4, 'sources_returned': 69, 'findings_kept': 5, 'dropped': {'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 2, 'date_outside_window': 0}, 'blocked_domain_sources': 0, 'seconds': 21.7}

findings kept: 5, new this run: 5


In [8]:
from IPython.display import Markdown, display

display(Markdown(state["report"]))

| Package | What happened | ID | Severity | Exploited | Action | Source |
|---|---|---|---|---|---|---|
| openssl (container base images) | New OpenSSL vulnerability affecting CRL distribution point caching causes disproportionate heap growth / DoS during TLS handshake (CVE-2026-35189) | CVE-2026-35189 | high | no | Apply the OpenSSL update that defers CRL distribution point extension processing to avoid large memory allocations. | [ibm.com](https://www.ibm.com/support/pages/security-bulletin-due-use-openssl-ibm-entirex-vulnerable-multiple-vulnerabilities-2) |
| zlib (container base images) | zlib 1.4 released fixing over a dozen memory-safety flaws including a gzwrite buffer overflow (CVE-2026-85091), an inflateGetDictionary out-of-bounds write of up to 8MB, inflateSync one-byte window overrun, inflatePrime out-of-bounds read, and deflatePrime pending-buffer overrun, plus minizip path bugs | CVE-2026-85091 | high | no | Upgrade zlib to version 1.4 and check applications/container images for bundled or statically-linked older zlib copies that also need updating. | [freenode.net](https://freenode.net/article/zlib-1-4-fixes-buffer-overflows-and-minizip-path-bugs) |
| glibc (container base images) | glibc dynamic loader TOCTOU race in $ORIGIN/DT_RPATH handling allows local privilege escalation when fs.protected_hardlinks is disabled (CVE-2026-86805) | CVE-2026-86805 | medium | no | Update glibc to the fixed version and ensure fs.protected_hardlinks sysctl is enabled on affected hosts. | [securityfocus.com](https://www.securityfocus.com/advisory/MGASA-2026-0581) |
| glibc (container base images) | Oracle Linux glibc advisories fix four CVEs including DNS decoding DoS, fopen-mode heap overflow, and tdelete out-of-bounds writes | CVE-2026-18374 | medium | no | Apply the glibc update resolving the DNS decoding, fopen-mode heap overflow, and tdelete out-of-bounds write issues. | [linuxsecurity.com](https://linuxsecurity.com/advisories/oracle/oracle8-elsa-2026-80798-glibc) |
| glibc (container base images) | Oracle Linux glibc advisory fixes stack overflow in wordexp tilde expansion and related decoding issues (CVE-2026-6368, CVE-2026-6791) | CVE-2026-6368 | medium | no | Apply the glibc update that fixes the wordexp tilde-expansion stack overflow and related decoding bug. | [linuxsecurity.com](https://linuxsecurity.com/advisories/oracle/oracle8-elsa-2026-76777-glibc) |

### Check the report against the date window and the blocklist

The branch statistics above count findings dropped by the evidence check, by reason. The
branches already drop anything outside the window; this cell re-checks the rows that survived,
independently of the graph, against the two rules that matter for an alert: the source was
published inside the window, and it isn't on the blocklist.

In [9]:
def check_rules(state):
    run_window_start = datetime.fromisoformat(state["window_start"])
    outside_window, undated, blocked = [], [], []
    for f in state["findings"]:
        published = datetime.fromisoformat(f["source_published"]) if f["source_published"] else None
        if published is None:
            undated.append(f["source_url"])
        elif published < run_window_start:
            outside_window.append((f["source_published"], f["source_url"]))
        if matches(f["source_url"], BLOCKLIST):
            blocked.append(f["source_url"])
    print(f"findings: {len(state['findings'])}")
    print(f"source published before the window: {len(outside_window)}")
    print(f"source date not parseable: {len(undated)}")
    print(f"from blocked domains: {len(blocked)}")
    print(f"dropped across branches: {[b['dropped'] for b in state['branch_stats']]}")
    print(f"blocked-domain sources returned by any search: {sum(b['blocked_domain_sources'] for b in state['branch_stats'])}")
    for f in state["findings"][:5]:
        print(f"   {f['source_published']} | {f['package']} | {f['source_url']}")


check_rules(state)

findings: 5
source published before the window: 0
source date not parseable: 0
from blocked domains: 0
dropped across branches: [{'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 0, 'date_outside_window': 0}, {'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 2, 'date_outside_window': 0}, {'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 2, 'date_outside_window': 0}]
blocked-domain sources returned by any search: 0
   2026-11-04T00:00:00+00:00 | openssl | https://www.ibm.com/support/pages/security-bulletin-due-use-openssl-ibm-entirex-vulnerable-multiple-vulnerabilities-2
   2026-11-09T16:58:00+00:00 | glibc | https://www.securityfocus.com/advisory/MGASA-2026-0581
   2026-11-04T12:57:00+00:00 | glibc | https://linuxsecurity.com/advisories/oracle/oracle8-elsa-2026-80798-glibc
   2026-11-04T02:15:00+00:00 | glibc | https://linuxsecurity.com/advisories/oracle/oracle8-elsa-2026-76777-glibc
   2026-11-05T20:03:00+00:00 | zlib |

## Step 6: Run the watch again to see only new findings

A scheduled watch runs every day, and the team shouldn't see the same advisory twice. The
store from the first run already holds every reported finding, so a second run reports only
findings it hasn't seen.

In [10]:
state_2, wall_2 = run_watch()
print(f"second run: {len(state_2['findings'])} findings found, {len(state_2['new_findings'])} new")
print(state_2["report"] if state_2["new_findings"] else "nothing new to report")

second run: 9 findings found, 7 new
| Package | What happened | ID | Severity | Exploited | Action | Source |
|---|---|---|---|---|---|---|
| axios (npm) | Compromised npm maintainer account published malicious axios 1.14.1 and 0.30.4 containing a postinstall RAT dependency (WAVESHAPER.V2), attributed to North Korea-nexus actor UNC1069 |  | critical | yes | Do not install axios 1.14.1 or 0.30.4; pin to 1.14.0 or earlier/0.30.3 or earlier, treat any host that installed them as compromised, rotate credentials, and clear npm/yarn/pnpm caches. | [orcarouter.ai](https://www.orcarouter.ai/blog/2026-03-30-axios-npm-compromised) |
| axios (npm) | Axios npm supply chain attack: stolen long-lived npm token bypassed OIDC trusted publishing to push malicious releases |  | critical | yes | Audit CI/CD and dev environments for the malicious axios releases, revert compromised systems, and enforce npm 2FA with revocation of legacy tokens. | [netrouting.com](https://netrouting.com/axios-npm-attack-ci-c

## Step 7 (optional): Run the watch through AgentCore Gateway

The Web Search API and the Web Search Tool on AgentCore Gateway search the same index.
Choose by who enforces the rules and who calls search:

- **Use the Web Search API**, as in Steps 1 to 6, when your own application calls search and
  enforces its own rules. It needs one IAM permission and nothing else.
- **Use AgentCore Gateway** when a platform team offers search to many agents or teams and wants
  AWS to apply one administrator source policy to every search. The policy sits on the gateway
  target and is hidden from the agent, which can't lift it. Use the gateway also when Model
  Context Protocol (MCP) clients such as Claude need search, or when you want web search next
  to other tools behind one MCP endpoint.

This step creates a gateway and a web search target, `security-sources`, that carries
`BLOCKLIST` as an administrator policy. Then it runs the same graph with a fresh store.

![The dependency watch through AgentCore Gateway](images/agentcore_web_search_supply_chain_watch.svg)

Two IAM grants, on two different principals:

- The **gateway's service role** needs `bedrock-agentcore:InvokeWebSearch` on
  `arn:aws:bedrock-agentcore:<region>:aws:tool/web-search.v1`, and a trust policy that lets
  `bedrock-agentcore.amazonaws.com` assume it. If the trust policy scopes `aws:SourceArn`,
  the region in it must be the gateway's region, or the first search fails with
  `Failed to obtain execution role credentials`.
- **Your own credentials** need `bedrock-agentcore:InvokeGateway` on the gateway, permission
  to create, read and delete AgentCore gateways and targets, and, if the next cell creates the
  service role, `iam:GetRole`, `iam:CreateRole`, `iam:PutRolePolicy` and `iam:PassRole`.

The first cell creates the service role if it doesn't exist yet, or uses the role in
`GATEWAY_ROLE_ARN` if you set one. The second cell creates the gateway and target, reuses them
if they already exist, and fails loudly if a reused target is on an older connector version.

In [11]:
GATEWAY_ROLE_NAME = "AgentCoreWebSearchGatewayRole"
iam = boto3.client("iam")
account_id = boto3.client("sts").get_caller_identity()["Account"]

if os.environ.get("GATEWAY_ROLE_ARN"):
    GATEWAY_ROLE_ARN = os.environ["GATEWAY_ROLE_ARN"]
else:
    try:
        GATEWAY_ROLE_ARN = iam.get_role(RoleName=GATEWAY_ROLE_NAME)["Role"]["Arn"]
    except iam.exceptions.NoSuchEntityException:
        trust = {
            "Version": "2012-10-17",
            "Statement": [{
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
                "Condition": {
                    "StringEquals": {"aws:SourceAccount": account_id},
                    "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/*"},
                },
            }],
        }
        GATEWAY_ROLE_ARN = iam.create_role(
            RoleName=GATEWAY_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust),
            Description="Lets AgentCore Gateway call AgentCore Web Search",
        )["Role"]["Arn"]
        iam.put_role_policy(
            RoleName=GATEWAY_ROLE_NAME, PolicyName="InvokeWebSearch",
            PolicyDocument=json.dumps({
                "Version": "2012-10-17",
                "Statement": [{
                    "Effect": "Allow",
                    "Action": "bedrock-agentcore:InvokeWebSearch",
                    "Resource": f"arn:aws:bedrock-agentcore:{REGION}:aws:tool/web-search.v1",
                }],
            }),
        )
        time.sleep(10)  # let the new role propagate before the gateway assumes it
print(f"gateway service role: {GATEWAY_ROLE_ARN.split('/')[-1]}")

gateway service role: AgentCoreWebSearchGatewayRole


In [12]:
from bedrock_agentcore.gateway.client import GatewayClient

GATEWAY_NAME = "langchain-supply-chain-watch"
TARGET_NAME = "security-sources"
CONNECTOR_VERSION = "1.3.0"
# 1.2.0 is the first connector version with request-level filters, which the date window
# relies on. A target reused from an earlier run may be on 1.2.0, which also works.
MIN_CONNECTOR_VERSION = (1, 2, 0)

gateway_client = GatewayClient(region_name=REGION)
gateway = gateway_client.get_gateway_by_name(GATEWAY_NAME)
if gateway is None:
    gateway = gateway_client.create_gateway_and_wait(
        name=GATEWAY_NAME,
        roleArn=GATEWAY_ROLE_ARN,
        authorizerType="AWS_IAM",
        protocolType="MCP",
    )
GATEWAY_ID = gateway["gatewayId"]

target = gateway_client.get_gateway_target_by_name(GATEWAY_ID, TARGET_NAME)
if target is None:
    target = gateway_client.create_web_search_target(
        gateway_identifier=GATEWAY_ID,
        name=TARGET_NAME,
        exclude_domains=BLOCKLIST,
        connector_version=CONNECTOR_VERSION,
    )
connector = target["targetConfiguration"]["mcp"]["connector"]
version = connector["source"].get("version") or "0"
if tuple(int(p) for p in version.split(".")) < MIN_CONNECTOR_VERSION:
    raise RuntimeError(f"{TARGET_NAME} is on connector {version}; the date window needs 1.2.0 or later")
print(f"{TARGET_NAME}: {target['status']}, connector {version}, "
      f"admin policy {connector['configurations'][0]['parameterValues']}")

security-sources: READY, connector 1.2.0, admin policy {'domainFilter': {'exclude': ['amazon.com']}}


In [13]:
# A fresh store, so this run reports its own findings instead of only what's new since Step 6.
gateway_watch = builder.compile(store=InMemoryStore())
gateway_state, gateway_wall = run_watch(gateway_watch, route="gateway")
print(f"gateway route: wall time {gateway_wall}s, findings kept: {len(gateway_state['findings'])}\n")
check_rules(gateway_state)

gateway route: wall time 19.9s, findings kept: 9

findings: 9
source published before the window: 0
source date not parseable: 0
from blocked domains: 0
dropped across branches: [{'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 1, 'date_outside_window': 0}, {'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 0, 'date_outside_window': 0}, {'url_not_returned': 0, 'package_not_in_source': 1, 'id_not_in_source': 0, 'date_outside_window': 0}]
blocked-domain sources returned by any search: 0
   2026-11-07T04:02:00+00:00 | urllib3 | https://techjacksolutions.com/scc-intel/ghsa-qccp-gfcp-xxvc-urllib3-sensitive-headers-forwarded-across-origins-in-proxied-low-level-redirec/
   2026-11-05T20:49:00+00:00 | urllib3 | https://sharedlicense.com/advisories/python-urllib3-security-update-almalinux-9-alsa-202690494/
   2026-11-10T00:00:00+00:00 | openssl | https://seclists.org/oss-sec/2026/q4/630
   2026-11-04T00:00:00+00:00 | openssl | https://www.ibm.com/supp

## Next steps

- **Schedule it.** Run the graph once a day, or more often for a tight window, and post the
  report wherever the team works.
- **Use a durable store.** Swap `InMemoryStore` for a store with exact key lookups, such as
  `DynamoDBStore` from `langgraph-checkpoint-aws`, so "already reported" survives between runs.
- **Host it on AgentCore Runtime.** The `agentcore_web_search_supply_chain_runtime` folder next
  to this notebook has the same graph as an AgentCore Runtime entrypoint, with the steps to
  deploy it using the [AgentCore CLI](https://github.com/aws/agentcore-cli). Runtime session state
  is ephemeral, so pair it with the durable store above.
- **Give each tenant its own policy.** On the gateway route, add one web search target per
  tenant, each with its own blocklist. `agentcore_web_search_langgraph_walkthrough.ipynb` shows
  that pattern.

## Clean up

Steps 1 to 6 create no AWS resources. If you added the Step 1 permission only for this
notebook, remove it from the identity you attached it to. If you ran Step 7, the gateway and
target are left in place unless you set `DELETE_GATEWAY = True`. The IAM role is left in place
either way, so other gateways can use it; delete it in the IAM console when you no longer need it.

In [14]:
DELETE_GATEWAY = False

if DELETE_GATEWAY and "GATEWAY_ID" in globals():
    target = gateway_client.get_gateway_target_by_name(GATEWAY_ID, TARGET_NAME)
    if target:
        gateway_client.delete_gateway_target_and_wait(gatewayIdentifier=GATEWAY_ID, targetId=target["targetId"])
    gateway_client.delete_gateway_and_wait(gatewayIdentifier=GATEWAY_ID)
    print(f"deleted {GATEWAY_ID}")
else:
    print("nothing deleted")

nothing deleted
